# 30 — Medical Image Segmentation with PyTorch and U-Net

Classification predicts one label per image. Segmentation predicts a label for every pixel.

We will study:

- Semantic segmentation masks
- Pixel-wise prediction
- Encoder–decoder networks
- U-Net
- Skip connections
- BCE and Dice losses
- IoU
- Synchronized image/mask augmentation
- Multi-class segmentation
- Ultrasound-specific concerns
- Failure analysis


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset,DataLoader
print("PyTorch:",torch.__version__)


# 1. Segmentation Shape

Binary segmentation:

$$
(N,C,H,W)\rightarrow(N,1,H,W)
$$

The output contains one logit per pixel.


In [ ]:
def make_sample(size=64,seed=0):
    g=torch.Generator().manual_seed(seed)
    image=torch.randn(1,size,size,generator=g)*0.08
    mask=torch.zeros(1,size,size)
    cy=size//2+int(torch.randint(-5,6,(1,),generator=g))
    cx=size//2+int(torch.randint(-5,6,(1,),generator=g))
    ry=10;rx=15
    yy=torch.arange(size).view(-1,1)
    xx=torch.arange(size).view(1,-1)
    ellipse=((yy-cy)/ry)**2+((xx-cx)/rx)**2<=1
    mask[0,ellipse]=1
    image=(image+0.8*mask).clamp(0,1)
    return image,mask


In [ ]:
image,mask=make_sample()
fig,axes=plt.subplots(1,2,figsize=(6,3))
axes[0].imshow(image.squeeze(),cmap="gray");axes[0].set_title("Image")
axes[1].imshow(mask.squeeze(),cmap="gray");axes[1].set_title("Mask")
for a in axes:a.axis("off")
plt.show()


# 2. U-Net Intuition

U-Net combines:

$$
\boxed{
Encoder+Decoder+Skip\ Connections
}
$$

Skip connections recover fine spatial detail lost during downsampling.


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self,cin,cout):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(cin,cout,3,padding=1),nn.ReLU(),
            nn.Conv2d(cout,cout,3,padding=1),nn.ReLU()
        )
    def forward(self,x): return self.net(x)


In [ ]:
class Down(nn.Module):
    def __init__(self,cin,cout):
        super().__init__()
        self.net=nn.Sequential(nn.MaxPool2d(2),DoubleConv(cin,cout))
    def forward(self,x): return self.net(x)

class Up(nn.Module):
    def __init__(self,cin,skip,out):
        super().__init__()
        self.up=nn.ConvTranspose2d(cin,out,2,stride=2)
        self.conv=DoubleConv(out+skip,out)
    def forward(self,x,s):
        x=self.up(x)
        if x.shape[-2:]!=s.shape[-2:]:
            x=F.interpolate(x,size=s.shape[-2:],mode="bilinear",align_corners=False)
        return self.conv(torch.cat([s,x],dim=1))


# 3. Complete Small U-Net


In [ ]:
class SmallUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1=DoubleConv(1,16)
        self.e2=Down(16,32)
        self.e3=Down(32,64)
        self.b=Down(64,128)
        self.u3=Up(128,64,64)
        self.u2=Up(64,32,32)
        self.u1=Up(32,16,16)
        self.out=nn.Conv2d(16,1,1)
    def forward(self,x):
        x1=self.e1(x);x2=self.e2(x1);x3=self.e3(x2);b=self.b(x3)
        x=self.u3(b,x3);x=self.u2(x,x2);x=self.u1(x,x1)
        return self.out(x)


In [ ]:
model=SmallUNet()
print(model(torch.randn(4,1,64,64)).shape)


# 4. BCEWithLogitsLoss

For binary segmentation, output raw logits and use:

```python
nn.BCEWithLogitsLoss()
```

Do not apply sigmoid before the loss.


# 5. Dice Coefficient

$$
\boxed{
Dice=\frac{2|P\cap T|}{|P|+|T|}
}
$$


In [ ]:
def dice_score(logits,target,threshold=0.5,eps=1e-6):
    pred=(torch.sigmoid(logits)>=threshold).float()
    inter=(pred*target).sum((1,2,3))
    denom=pred.sum((1,2,3))+target.sum((1,2,3))
    return ((2*inter+eps)/(denom+eps)).mean().item()


# 6. Soft Dice Loss


In [ ]:
class SoftDiceLoss(nn.Module):
    def forward(self,logits,target,eps=1e-6):
        p=torch.sigmoid(logits)
        inter=(p*target).sum((1,2,3))
        denom=p.sum((1,2,3))+target.sum((1,2,3))
        return 1-((2*inter+eps)/(denom+eps)).mean()


# 7. BCE + Dice


In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self,dice_weight=1.0):
        super().__init__()
        self.bce=nn.BCEWithLogitsLoss()
        self.dice=SoftDiceLoss()
        self.w=dice_weight
    def forward(self,logits,target):
        return self.bce(logits,target)+self.w*self.dice(logits,target)


# 8. IoU / Jaccard

$$
\boxed{
IoU=\frac{|P\cap T|}{|P\cup T|}
}
$$


In [ ]:
def iou_score(logits,target,threshold=0.5,eps=1e-6):
    pred=(torch.sigmoid(logits)>=threshold).float()
    inter=(pred*target).sum((1,2,3))
    union=pred.sum((1,2,3))+target.sum((1,2,3))-inter
    return ((inter+eps)/(union+eps)).mean().item()


# 9. Dataset and DataLoader


In [ ]:
class SegDataset(Dataset):
    def __init__(self,n=200): self.n=n
    def __len__(self): return self.n
    def __getitem__(self,i): return make_sample(seed=1000+i)

train_ds=SegDataset(160);val_ds=SegDataset(40)
train_loader=DataLoader(train_ds,batch_size=16,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=32)


# 10. Training Loop


In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_epoch(model,loader,criterion,optimizer):
    model.train();total=0;n=0
    for x,y in loader:
        x=x.to(device);y=y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits=model(x);loss=criterion(logits,y)
        loss.backward();optimizer.step()
        total+=loss.item()*x.size(0);n+=x.size(0)
    return total/n


# 11. Validation


In [ ]:
def evaluate(model,loader,criterion):
    model.eval();loss_sum=0;n=0;ds=[];js=[]
    with torch.inference_mode():
        for x,y in loader:
            x=x.to(device);y=y.to(device)
            logits=model(x);loss=criterion(logits,y)
            loss_sum+=loss.item()*x.size(0);n+=x.size(0)
            ds.append(dice_score(logits,y));js.append(iou_score(logits,y))
    return {"loss":loss_sum/n,"dice":sum(ds)/len(ds),"iou":sum(js)/len(js)}


# 12. Synchronized Image/Mask Augmentation

Geometric transforms must be identical for image and mask.


In [ ]:
def horizontal_flip_pair(image,mask):
    return torch.flip(image,[2]),torch.flip(mask,[2])


# 13. Mask Interpolation

When resizing discrete class masks, use nearest-neighbor interpolation. Bilinear interpolation creates invalid intermediate class values.


# 14. Multi-Class Segmentation

For $K$ classes:

$$
Logits:(N,K,H,W)
$$

$$
Targets:(N,H,W)
$$

Use `CrossEntropyLoss`.


# 15. Class Imbalance

Background often dominates foreground. Pixel accuracy can therefore be misleading.

Report Dice/IoU and class-specific performance.


# 16. Empty Masks

Some images may contain no target. Define explicitly how Dice is calculated in this case.


# 17. Boundary Metrics

Overlap metrics do not fully characterize boundary quality.

Other measures include:

- Hausdorff distance
- Average surface distance
- Boundary F-score


# 18. Ultrasound-Specific Challenges

- Speckle
- Weak boundaries
- Acoustic shadowing
- Missing edges
- Operator/view variability


# 19. Annotation Variability

Ground-truth boundaries may vary across annotators. Consider multiple experts and inter-rater agreement.


# 20. Segmentation + Classification

One workflow:

$$
Image\rightarrow Segmentation\rightarrow ROI\rightarrow Classification
$$

But segmentation errors may propagate downstream.


# 21. Common Mistakes

- Sigmoid before `BCEWithLogitsLoss`
- Unsynchronized image/mask augmentation
- Bilinear interpolation for class masks
- Reporting only pixel accuracy
- Leakage from manually defined ROIs


# 22. Exercises

1. Implement binary mask loading.
2. Build U-Net blocks.
3. Verify skip shapes.
4. Implement Dice.
5. Implement soft Dice loss.
6. Implement IoU.
7. Compare BCE vs BCE+Dice.
8. Create synchronized augmentation.
9. Explain empty-mask handling.
10. Inspect segmentation failures visually.


# 23. Key Takeaways

Segmentation predicts:

$$
\boxed{One\ Label\ Per\ Pixel}
$$

U-Net provides:

$$
\boxed{Encoder+Decoder+Skip\ Connections}
$$

For binary segmentation, BCE+Dice is a strong baseline.


# Next Notebook

# 31 — Self-Supervised and Contrastive Learning for Medical Images

In the next notebook, we will study unlabeled-data representation learning, SimCLR, contrastive loss, linear probing, and ultrasound-specific SSL.
